#Librerias

In [1]:
import pandas as pd
import os

# Librerías de manipulación de datos
import pandas as pd
import numpy as np

# Opciones de Pandas
pd.set_option('display.max_columns', None)

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import plotly.express as px

# Librerías para estadística y pruebas
import scipy.stats as stats
from scipy.stats import gaussian_kde, kstest, norm, shapiro
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Librerías de machine learning y preprocesamiento
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Librerías para métricas de modelos
from sklearn.model_selection import train_test_split, GridSearchCV
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error,  mean_absolute_error, mean_absolute_percentage_error, r2_score, classification_report
import math

#Carga de datos

In [6]:
os.environ['KAGGLE_CONFIG_DIR'] = '.'

In [7]:
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 2.51GB/s]


In [8]:
!unzip udea-ai-4-eng-20252-pruebas-saber-pro-colombia

Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  inflating: submission_example.csv  
  inflating: test.csv                
  inflating: train.csv               


In [9]:
df_train=pd.read_csv('train.csv')

# Datos faltantes

En el análisis exploratorio se encontró que el dataset de entrenamiento está conformado por 692500 filas y 21 columnas (variables). El dataset no tiene columnas repetidas con el mismo nombre pero si tiene repetida la columna "F_TIENEINTERNET" como "F_TIENEINTERNET.1" esto debido a que ambas columnas tienen la misma cantidad de datos con SI y NO y tambien la misma cantidad de faltantes. No tiene filas duplicadas pero si tiene datos faltantes; entre las columnas con mayor proporción de faltantes se encuentran 'F_TIENEAUTOMOVIL', 'F_TIENELAVADORA' y 'F_TIENE COMPUTADOR' que representan más del 50% de los faltantes otras como 'E_VALORMATRICULAUNIVERSIDAD' y 'E_PAGOMATRICULAPROPIO' no representan ni el 1% de los faltantes. Se modificaron los tipos de datos para 'ID' y para 'PERIODO_ACADEMICO' puesto que estas variables aunque sean números están expresando categorías. El dataset solo tiene 4 variables númericas que son 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4'.

In [10]:
# Seleccionar solo las columnas de interés
cols = [
    'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
       'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE',
       'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PAGOMATRICULAPROPIO',
       'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE'
]

# Crear dataset de faltantes
faltantes_df = pd.DataFrame({
    "Columna": cols,
    "Faltantes": df_train[cols].isnull().sum().values,
    "Porcentaje (%)": (df_train[cols].isnull().mean().values * 100).round(2)
})

faltantes_df

,Columna,Faltantes,Porcentaje (%)
0,E_VALORMATRICULAUNIVERSIDAD,6287,0.91
1,E_HORASSEMANATRABAJA,30857,4.46
2,F_ESTRATOVIVIENDA,32137,4.64
3,F_TIENEINTERNET,26629,3.85
4,F_EDUCACIONPADRE,23178,3.35
5,F_TIENELAVADORA,39773,5.74
6,F_TIENEAUTOMOVIL,43623,6.30
7,E_PAGOMATRICULAPROPIO,6498,0.94
8,F_TIENECOMPUTADOR,38103,5.50
9,F_EDUCACIONMADRE,23664,3.42


In [11]:
df_train= df_train.drop(columns=['F_TIENEINTERNET.1'],  errors='ignore') #Eliminación de columna repetida

In [12]:
df_train['ID']=df_train['ID'].astype('object')
df_train['PERIODO_ACADEMICO']=df_train['PERIODO_ACADEMICO'].astype('object')

Algunas variables no son ordinales como:

'F_TIENEINTERNET','F_TIENELAVADORA','F_TIENEAUTOMOVIL','F_TIENECOMPUTADOR','E_PAGOMATRICULAPROPIO',

es decir, sus categorias no representan un orden, los valores faltantes de estas serán reemplazados con la categoria 'otro'. La variable 'F_ESTRATOVIVIENDA' tiene una categoria llamada 'Sin Estrato' aquí se añadirán todos los valores faltantes de dicha variable. Las variables restantes serán tratadas posteriormente luego de convertirlas a categoricas para ayudar al modelo a interpretar la ordinalidad de cada una.

In [13]:
for col in ['F_TIENEINTERNET', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'F_TIENECOMPUTADOR','E_PAGOMATRICULAPROPIO']:
    df_train[col] = df_train[col].fillna('otro')

In [14]:
df_train['F_ESTRATOVIVIENDA']=df_train['F_ESTRATOVIVIENDA'].fillna('Sin Estrato')

In [15]:
# Lista de columnas y títulos
cols = [
    ("F_TIENECOMPUTADOR", "Acceso a computador"),
    ("E_PRGM_DEPARTAMENTO", "Departamento donde cursó la carrera"),
    ("E_HORASSEMANATRABAJA", "Horas trabajadas a la semana"),
    ("F_ESTRATOVIVIENDA", "Estrato vivienda"),
    ("F_TIENEINTERNET", "Acceso a internet"),
    ("F_TIENELAVADORA", "Acceso a lavadora"),
    ("F_TIENEAUTOMOVIL", "Acceso a automóvil propio"),
    ("E_PRIVADO_LIBERTAD", "Privación de la libertad"),
    ("E_PAGOMATRICULAPROPIO", "Costeo propio de matrícula"),
    ("F_EDUCACIONPADRE", "Nivel educativo del padre"),
    ("F_EDUCACIONMADRE", "Nivel educativo de la madre"),
    ("E_VALORMATRICULAUNIVERSIDAD", "$ matrícula universidad")
]

# Crear figura con subplots
fig = make_subplots(
    rows=3, cols=4,
    subplot_titles=[f"<b>{title}<b>" for _, title in cols],
    vertical_spacing=0.25,
    horizontal_spacing=0.1
)

# Agregar gráficos en un ciclo
for i, (col, title) in enumerate(cols, start=1):
    # Calcular fila y columna
    row = (i - 1) // 4 + 1
    col_pos = (i - 1) % 4 + 1

    # Agrupar y contar
    base = (
        df_train.groupby([col])[["ID"]]
        .count()
        .reset_index()
        .sort_values("ID", ascending=False)
        .rename(columns={"ID": "count"})
    )

    # Crear gráfico de barras
    fig_bar = px.bar(base, x=col, y="count", barmode="group")

    # Añadir trazas al subplot correspondiente
    for trace in fig_bar.data:
        fig.add_trace(trace, row=row, col=col_pos)

# Ajustes de layout
fig.update_layout(height=1000, width=1500)

# Mostrar
fig.show()


In [16]:
df_train = df_train.drop(columns=['ID'], errors='ignore')

# Feature engineering

## Creación de variables

**CATEGORIA_PROGRAMA**

Esta variable agrupa los distintos programas académicos (más de 900 originalmente) en categorías temáticas o áreas de conocimiento, como Ingenierías, Ciencias Sociales, Administración y Economía, Educación, Ciencias de la Salud, Artes y Comunicación, entre otras. Su objetivo es representar el campo disciplinar de formación del estudiante sin perder información relevante, pero reduciendo la alta cardinalidad original. Esto facilita al modelo identificar patrones de desempeño por área académica.

In [17]:
def categorizar_programa(x):
    x = str(x).upper()

    if any(word in x for word in ['INGENIER', 'SISTEMAS', 'SOFTWARE', 'COMPUTACIÓN', 'ELECTR', 'INDUSTRIAL', 'CIVIL', 'MECANIC', 'TECNOLOG','ARQUI','COMPU','URBAN','CONSTR','OCEANO','DISENO']):
        return 'INGENIERÍAS Y CIENCIAS APLICADAS'
    elif any(word in x for word in ['MEDIC', 'ENFERM', 'SALUD', 'FISIOTERAP', 'BIOQUIM', 'NUTRIC', 'ODONTO','INTRU','INSTRU','BACTER','MICROB','OPTO','TERAPIA','FONO','FARMACE','FARMA','ENFERM','VETERIN']):
        return 'CIENCIAS DE LA SALUD'
    elif any(word in x for word in ['SOCIOLOG', 'PSICOLOG', 'FILOSOF', 'HISTOR', 'ANTROPOLOG', 'TEOLOG', 'BIBLIC', 'POLITIC', 'GOBIERNO','DERECHO','TRABAJO','POLÍTICA','PSICÓLOGA','INTERNACIONALES','JURIS','ARQUEO','ARCHI','GERONT']):
        return 'CIENCIAS SOCIALES Y HUMANAS'
    elif any(word in x for word in ['ADMINISTR', 'ECONOM', 'NEGOC', 'FINANZ', 'COMERC', 'EMPRES', 'MARKETING', 'GERENCIA','CONTAD']):
        return 'ADMINISTRACIÓN Y ECONOMÍA'
    elif any(word in x for word in ['LICENCIAT', 'PEDAGOG', 'DOCEN', 'EDUCAC', 'ENSEÑAN','TRADUCC','LINGUISTICA']):
        return 'EDUCACIÓN'
    elif any(word in x for word in ['MATEM', 'FISIC', 'QUIMIC', 'BIOLOG', 'GEOLOG', 'ECOLOG', 'ASTRONOM','FÍSICA','QUÍMICA','ESTADISTICA','ESTADÍSTICA','ZOO']):
        return 'CIENCIAS EXACTAS Y NATURALES'
    elif any(word in x for word in ['ARTE', 'MUSICA', 'TEATRO', 'CINE', 'DISEÑ', 'PUBLICID', 'COMUNIC', 'LITERAT', 'FILOL', 'LENGUA','MÚSICA','DANZA','BANDA','LITERARIA','LITERARIOS','FOTOGRAFÍA','AUDIOVISUALES','PRODUCCIÓN','ANIMACIÓN','CULTURAL']):
        return 'ARTES, COMUNICACIÓN Y HUMANIDADES'
    elif any(word in x for word in ['TURIS', 'HOTEL', 'GASTRON', 'CULINAR']):
        return 'TURISMO Y HOTELERÍA'
    elif any(word in x for word in ['CRIMINAL', 'INVESTIGAC', 'POLIC', 'MILITAR', 'NAVA', 'SEGURIDAD']):
        return 'CRIMINALÍSTICA Y SEGURIDAD'
    elif any(word in x for word in ['DEPORTE', 'DEPORTIVO','DEPORTIVA']):
        return 'DEPORTE'
    elif any(word in x for word in ['MERCADEO', 'MERCADOLOGIA','PLANEACION']):
        return 'NEGOCIOS Y MERCADEO'
    else:
        return 'OTRAS'

# Aplicar la función
df_train['CATEGORIA_PROGRAMA'] = df_train['E_PRGM_ACADEMICO'].apply(categorizar_programa)


In [18]:
df_train['CATEGORIA_PROGRAMA'].unique()

array(['CIENCIAS DE LA SALUD', 'CIENCIAS SOCIALES Y HUMANAS',
       'ARTES, COMUNICACIÓN Y HUMANIDADES', 'ADMINISTRACIÓN Y ECONOMÍA',
       'INGENIERÍAS Y CIENCIAS APLICADAS', 'TURISMO Y HOTELERÍA',
       'EDUCACIÓN', 'CIENCIAS EXACTAS Y NATURALES', 'DEPORTE',
       'NEGOCIOS Y MERCADEO', 'CRIMINALÍSTICA Y SEGURIDAD', 'OTRAS'],
      dtype=object)

**REGION**

Esta variable clasifica a los estudiantes según la región geográfica en la que se encuentra su programa académico. A partir del departamento (E_PRGM_DEPARTAMENTO), se agruparon los territorios de Colombia en grandes regiones: Andina, Caribe, Pacífica, Orinoquía y Amazónica. Esta agrupación permite capturar diferencias contextuales en la calidad de la educación, acceso a recursos o infraestructura entre zonas del país, las cuales pueden influir significativamente en el rendimiento académico.

In [19]:
region_map = {
    'ANTIOQUIA': 'Andina', 'CUNDINAMARCA': 'Andina', 'BOGOTÁ': 'Andina',
    'BOYACA': 'Andina', 'SANTANDER': 'Andina', 'NORTE SANTANDER': 'Andina',
    'CALDAS': 'Andina', 'QUINDIO': 'Andina', 'RISARALDA': 'Andina', 'TOLIMA': 'Andina',
    'HUILA': 'Andina',
    'ATLANTICO': 'Caribe', 'BOLIVAR': 'Caribe', 'CESAR': 'Caribe', 'CORDOBA': 'Caribe',
    'LA GUAJIRA': 'Caribe', 'MAGDALENA': 'Caribe', 'SUCRE': 'Caribe', 'SAN ANDRES': 'Caribe',
    'VALLE': 'Pacífica', 'CAUCA': 'Pacífica', 'NARIÑO': 'Pacífica', 'CHOCO': 'Pacífica',
    'META': 'Orinoquía', 'CASANARE': 'Orinoquía', 'ARAUCA': 'Orinoquía',
    'AMAZONAS': 'Amazónica', 'PUTUMAYO': 'Amazónica', 'CAQUETA': 'Amazónica',
    'GUAVIARE': 'Amazónica', 'VAUPES': 'Amazónica'
}

df_train['REGION'] = df_train['E_PRGM_DEPARTAMENTO'].map(region_map)


**AÑO**

Esta variable representa el año de presentación o cohorte académica del estudiante, extraído del campo PERIODO_ACADEMICO. Captura el componente temporal del dataset y permite observar posibles tendencias o cambios en el desempeño a lo largo del tiempo, relacionados con políticas educativas, reformas curriculares o variaciones en la cobertura del examen

In [20]:
df_train['AÑO'] = df_train['PERIODO_ACADEMICO'].astype(str).str[:4].astype(int)

**INDICE_SOCIOECO**

El índice socioeconómico es un indicador compuesto que resume el nivel de bienestar del hogar del estudiante. Se calculó sumando variables binarias sobre la tenencia de bienes y servicios básicos (F_TIENEINTERNET, F_TIENECOMPUTADOR, F_TIENEAUTOMOVIL, F_TIENELAVADORA). Un valor de 4 refleja un entorno con mayor acceso a recursos tecnológicos y económicos. Esta variable permite representar de manera compacta el contexto socioeconómico, fuertemente correlacionado con el rendimiento educativo.

In [21]:
df_train['INDICE_SOCIOECO'] = (
    (df_train['F_TIENEINTERNET'] == 'Si').astype(int) +
    (df_train['F_TIENECOMPUTADOR'] == 'Si').astype(int) +
    (df_train['F_TIENEAUTOMOVIL'] == 'Si').astype(int) +
    (df_train['F_TIENELAVADORA'] == 'Si').astype(int)
)

**VALOR_MATRICULA_NUM**

Corresponde a una versión numérica del rango de valor de matrícula (E_VALORMATRICULAUNIVERSIDAD), expresada mediante el punto medio de cada intervalo monetario. Por ejemplo, “Entre 1 millón y menos de 2.5 millones” se convierte en 1,750,000. Esta transformación permite utilizar la variable como un indicador cuantitativo del nivel económico institucional o de acceso financiero del estudiante, asociado al poder adquisitivo y a los recursos disponibles durante su formación.

In [22]:
matricula_map = {
    'Menos de 500 mil': 250_000,
    'Entre 500 mil y menos de 1 millón': 750_000,
    'Entre 1 millón y menos de 2.5 millones': 1_750_000,
    'Entre 2.5 millones y menos de 4 millones': 3_250_000,
    'Entre 4 millones y menos de 5.5 millones': 4_750_000,
    'Entre 5.5 millones y menos de 7 millones': 6_250_000,
    'Más de 7 millones': 8_000_000,
    'No pagó matrícula': 0,
}

df_train['VALOR_MATRICULA_NUM'] = df_train['E_VALORMATRICULAUNIVERSIDAD'].map(matricula_map)
df_train['VALOR_MATRICULA_NUM']=df_train['VALOR_MATRICULA_NUM'].fillna(df_train['VALOR_MATRICULA_NUM'].median())

**EDU_PADRE_NUM Y EDU_MADRE_NUM**

Esta variable convierte los niveles educativos del padre (F_EDUCACIONPADRE) en una escala numérica ordinal de 0 a 9, donde 0 indica “Ninguno” y 9 “Postgrado”. Permite representar la educación paterna en un formato que conserva el orden jerárquico y facilita su interpretación en modelos predictivos. Esta información es clave para comprender el entorno educativo familiar. De forma análoga, EDU_MADRE_NUM representa el nivel educativo de la madre en escala ordinal. Junto con EDU_PADRE_NUM, permite identificar diferencias en el nivel de formación parental, que pueden influir en el acompañamiento académico del estudiante. Estas dos variables son esenciales para estimar el efecto del capital cultural y educativo del hogar en el desempeño global.

In [23]:
edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9,
    'No sabe': 0,
    'No Aplica': 0,
}

df_train['EDU_PADRE_NUM'] = df_train['F_EDUCACIONPADRE'].map(edu_map)
df_train['EDU_MADRE_NUM'] = df_train['F_EDUCACIONMADRE'].map(edu_map)

moda_padre = df_train['EDU_PADRE_NUM'].mode()[0]
moda_madre = df_train['EDU_MADRE_NUM'].mode()[0]

df_train['EDU_PADRE_NUM']=df_train['EDU_PADRE_NUM'].fillna(moda_padre)
df_train['EDU_MADRE_NUM']=df_train['EDU_MADRE_NUM'].fillna(moda_madre)

**EDU_PROMEDIO_FAM**

El promedio educativo familiar se obtuvo calculando la media entre los niveles educativos del padre y la madre (EDU_PADRE_NUM y EDU_MADRE_NUM). Esta variable cuantifica el capital educativo del hogar y funciona como un indicador del entorno formativo y cultural del estudiante. Diversos estudios muestran que un mayor nivel educativo de los padres está positivamente asociado al desempeño académico de los hijos.

**ALGUNO_SUPERIOR**

Es una variable binaria (0 o 1) que indica si al menos uno de los padres posee educación superior (técnica, tecnológica o profesional). Se construyó a partir de los valores numéricos de educación del padre y la madre. Este indicador captura la exposición del estudiante a un entorno familiar con experiencia universitaria, lo cual suele influir en la motivación, hábitos de estudio y apoyo académico disponible.

In [24]:
df_train['EDU_PROMEDIO_FAM'] = df_train[['EDU_PADRE_NUM', 'EDU_MADRE_NUM']].mean(axis=1)

df_train['ALGUNO_SUPERIOR'] = np.where(
    (df_train['EDU_PADRE_NUM'] >= 6) | (df_train['EDU_MADRE_NUM'] >= 6), 1, 0
)

**HORAS_TRABAJO_NUM**

Representa el número estimado de horas de trabajo semanal del estudiante, transformando las categorías de E_HORASSEMANATRABAJA (“Entre 11 y 20 horas”, “Más de 30 horas”, etc.) en valores numéricos ordinales aproximados (por ejemplo, 0, 5, 15, 25, 35). Esta variable refleja el grado de carga laboral del estudiante, que puede afectar su dedicación al estudio y, por tanto, su rendimiento en las pruebas.

In [25]:
horas_map = {
    '0': 0,
    'Menos de 10 horas': 5,
    'Entre 11 y 20 horas': 15,
    'Entre 21 y 30 horas': 25,
    'Más de 30 horas': 35
}

df_train['HORAS_TRABAJO_NUM'] = df_train['E_HORASSEMANATRABAJA'].map(horas_map)
moda_horas = df_train['HORAS_TRABAJO_NUM'].mode()[0]
df_train['HORAS_TRABAJO_NUM']=df_train['HORAS_TRABAJO_NUM'].fillna(moda_horas)

In [26]:
df_train[['REGION', 'CATEGORIA_PROGRAMA', 'VALOR_MATRICULA_NUM',
      'INDICE_SOCIOECO', 'AÑO', 'EDU_PROMEDIO_FAM', 'ALGUNO_SUPERIOR', 'HORAS_TRABAJO_NUM','EDU_PADRE_NUM','EDU_MADRE_NUM']].head()


,REGION,CATEGORIA_PROGRAMA,VALOR_MATRICULA_NUM,INDICE_SOCIOECO,AÑO,EDU_PROMEDIO_FAM,ALGUNO_SUPERIOR,HORAS_TRABAJO_NUM,EDU_PADRE_NUM,EDU_MADRE_NUM
0,Andina,CIENCIAS DE LA SALUD,6250000.0,4,2021,7.0,1,5.0,5.0,9.0
1,Caribe,CIENCIAS SOCIALES Y HUMANAS,3250000.0,2,2021,5.5,1,0.0,6.0,5.0
2,Andina,"ARTES, COMUNICACIÓN Y HUMANIDADES",3250000.0,2,2020,4.0,0,35.0,4.0,4.0
3,Andina,ADMINISTRACIÓN Y ECONOMÍA,4750000.0,3,2019,2.0,0,0.0,0.0,4.0
4,Andina,CIENCIAS SOCIALES Y HUMANAS,3250000.0,4,2021,2.0,0,25.0,2.0,2.0


## Eliminación de variables

In [27]:
df_clean=df_train.copy()

In [28]:
cols_drop = [
    'PERIODO_ACADEMICO',
    'E_PRGM_ACADEMICO',
    'E_PRGM_DEPARTAMENTO',
    'E_VALORMATRICULAUNIVERSIDAD',
    'E_HORASSEMANATRABAJA',
    'F_TIENEINTERNET',
    'F_EDUCACIONPADRE',
    'F_TIENELAVADORA',
    'F_TIENEAUTOMOVIL',
    'F_TIENECOMPUTADOR',
    'F_EDUCACIONMADRE',
    'F_ESTRATOVIVIENDA',
    'E_PRIVADO_LIBERTAD',
    'F_ESTRATOVIVIENDA',
    'E_PAGOMATRICULAPROPIO'
]
df_model = df_clean.drop(columns=cols_drop, errors='ignore')


In [29]:
df_model_2 = pd.get_dummies(df_model, columns=['CATEGORIA_PROGRAMA', 'REGION'], drop_first=True)

In [30]:
from sklearn.preprocessing import StandardScaler

# Columnas numéricas a escalar
num_cols = [
    'AÑO', 'VALOR_MATRICULA_NUM', 'INDICE_SOCIOECO',
    'EDU_PADRE_NUM', 'EDU_MADRE_NUM', 'EDU_PROMEDIO_FAM',
    'HORAS_TRABAJO_NUM'
]

scaler = StandardScaler()
df_model_2[num_cols] = scaler.fit_transform(df_model_2[num_cols])


In [31]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_model_2['RENDIMIENTO_GLOBAL'] = le.fit_transform(df_model_2['RENDIMIENTO_GLOBAL'])


In [32]:
X = df_model_2.drop(columns=['RENDIMIENTO_GLOBAL'])
y = df_model_2['RENDIMIENTO_GLOBAL']

## Métodos de filtrado

In [33]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X, y)

# Ver las variables más relevantes
feature_scores = pd.DataFrame({
    'Variable': X.columns,
    'F-Score': selector.scores_
}).sort_values(by='F-Score', ascending=False)

feature_scores.head(15)


,Variable,F-Score
9,EDU_PROMEDIO_FAM,23881.838181
8,EDU_MADRE_NUM,20903.916944
6,VALOR_MATRICULA_NUM,18596.698870
7,EDU_PADRE_NUM,16437.718334
0,INDICADOR_1,15660.913285
10,ALGUNO_SUPERIOR,14309.799956
5,INDICE_SOCIOECO,9510.702425
1,INDICADOR_2,9154.888284
19,CATEGORIA_PROGRAMA_INGENIERÍAS Y CIENCIAS APLI...,5051.147864
3,INDICADOR_4,4305.976143


## Métodos Wrapper

In [36]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': rf.feature_importances_
}).sort_values(by='Importancia', ascending=False)

importances.head(10)


,Variable,Importancia
0,INDICADOR_1,0.147843
1,INDICADOR_2,0.145659
3,INDICADOR_4,0.142571
2,INDICADOR_3,0.141811
6,VALOR_MATRICULA_NUM,0.068687
9,EDU_PROMEDIO_FAM,0.056580
4,AÑO,0.046102
7,EDU_PADRE_NUM,0.045195
11,HORAS_TRABAJO_NUM,0.045065
8,EDU_MADRE_NUM,0.043199


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe = RFE(estimator=LogisticRegression(max_iter=500), n_features_to_select=10)
rfe.fit(X, y)

selected = pd.DataFrame({
    'Variable': X.columns,
    'Seleccionada': rfe.support_
})
selected[selected['Seleccionada'] == True]
